In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np

data = pd.read_csv("weather_data.csv")

print(data.columns)# Convert DATE to actual datetime
data['DATE'] = pd.to_datetime(data['DATE'])

# Sort chronologically
data = data.sort_values('DATE')

# Remove columns we don't need
data = data.drop(columns=['STATION', 'NAME', 'SNWD'])

print(data.head())
data['TMAX'] = (data['TMAX'] - 32) * 5/9
data['TMIN'] = (data['TMIN'] - 32) * 5/9
print(data.head())
print(data.describe())
features = data[['PRCP', 'TMAX', 'TMIN']].values
X = []
y = []

sequence_length = 14

for i in range(len(features) - sequence_length):

    X.append(
        features[i:i + sequence_length]
    )

    y.append(
        features[i + sequence_length, 1]
    )
print(X[0])
print(y[0])

X = np.array(X)
y = np.array(y)

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

y = y.reshape(-1, 1)

print(X.shape)
print(y.shape)
train_size = int(0.70 * len(X))

val_size = int(0.15 * len(X))

X_train = X[:train_size]
y_train = y[:train_size]

X_val = X[train_size:train_size + val_size]
y_val = y[train_size:train_size + val_size]

X_test = X[train_size + val_size:]
y_test = y[train_size + val_size:]

print("Training:")
print(X_train.shape)
print(y_train.shape)

print("Validation:")
print(X_val.shape)
print(y_val.shape)

print("Testing:")
print(X_test.shape)
print(y_test.shape)

X_mean = X_train.mean(dim=(0, 1), keepdim=True)
X_std = X_train.std(dim=(0, 1), keepdim=True)

X_train = (X_train - X_mean) / X_std
X_val = (X_val - X_mean) / X_std
X_test = (X_test - X_mean) / X_std

y_mean = y_train.mean()
y_std = y_train.std()

y_train = (y_train - y_mean) / y_std
y_val = (y_val - y_mean) / y_std
y_test = (y_test - y_mean) / y_std

from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

trainloader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

valloader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

testloader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [ ]:
class WeatherLSTM(nn.Module):
    def __init__(self):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=3,
            hidden_size=64,
            num_layers=1,
            batch_first=True
        )

        self.fc = nn.Linear(64, 1)

    def forward(self, x):

        output, (hidden, cell) = self.lstm(x)

        x = hidden[-1]

        x = self.fc(x)

        return x
lstm_model = WeatherLSTM()

print(lstm_model)   
criterion = nn.MSELoss()

lstm_optimizer = torch.optim.Adam(
    lstm_model.parameters(),
    lr=0.001
)
epochs = 50

for epoch in range(epochs):

    # TRAINING
    lstm_model.train()

    train_loss = 0.0

    for X_batch, y_batch in trainloader:

        predictions = lstm_model(X_batch)

        loss = criterion(predictions, y_batch)

        lstm_optimizer.zero_grad()

        loss.backward()

        lstm_optimizer.step()

        train_loss += loss.item()

    train_loss /= len(trainloader)


    # VALIDATION
    lstm_model.eval()

    val_loss = 0.0

    with torch.no_grad():

        for X_batch, y_batch in valloader:

            predictions = lstm_model(X_batch)

            loss = criterion(predictions, y_batch)

            val_loss += loss.item()

    val_loss /= len(valloader)


    print(
        f"Epoch {epoch+1}/{epochs} "
        f"Train Loss: {train_loss:.4f} "
        f"Validation Loss: {val_loss:.4f}"
    )
    lstm_model.eval()

lstm_predictions = []
lstm_actual = []

with torch.no_grad():

    for X_batch, y_batch in testloader:

        predictions = lstm_model(X_batch)

        lstm_predictions.append(predictions)
        lstm_actual.append(y_batch)
lstm_predictions = torch.cat(lstm_predictions)
lstm_actual = torch.cat(lstm_actual)
lstm_predictions_real = (
    lstm_predictions * y_std + y_mean
)

lstm_actual_real = (
    lstm_actual * y_std + y_mean
)
lstm_mae = torch.mean(
    torch.abs(
        lstm_predictions_real - lstm_actual_real
    )
)

lstm_rmse = torch.sqrt(
    torch.mean(
        (lstm_predictions_real - lstm_actual_real) ** 2
    )
)

print("LSTM MAE:", lstm_mae.item())
print("LSTM RMSE:", lstm_rmse.item())